# Demo：只用《爱丽丝》——HMM 词性 / 生成 vs N-gram 生成

**语料仅一份**：`notebooks/data/stories/alice_in_wonderland.txt`  
（《Alice's Adventures in Wonderland》，Project Gutenberg #11）。不混 Brown 或其它书。

| 模型 | 训练信号 | 本 notebook 用途 |
|---|---|---|
| HMM | 自动词性 + 词 | 词性预测；按标签链续写 |
| N-gram | 纯词序列 | 按最近 $N-1$ 个词续写；**N 与句子长度可调** |

生成区支持：前缀词、句子长度、N-gram 的 N、条数、种子；可多阶并排对照。


In [1]:
from pathlib import Path
import sys

import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output

ROOT = Path.cwd()
if ROOT.name != "notebooks":
    if (ROOT / "notebooks").is_dir():
        ROOT = ROOT / "notebooks"
sys.path.insert(0, str(ROOT))

from hmm_demo import HiddenMarkovModel, NgramLM, load_plain_story_tagged

print("imports ok")

imports ok


## 1. 加载 Alice（唯一语料）

In [2]:
tagged_sents = load_plain_story_tagged()  # 仅 Alice；词性为 NLTK 自动标注
word_sents = [[w for w, _ in sent] for sent in tagged_sents]

rng = np.random.default_rng(0)
idx = rng.permutation(len(tagged_sents))
cut = int(0.9 * len(tagged_sents))
train_tagged = [tagged_sents[i] for i in idx[:cut]]
test_tagged = [tagged_sents[i] for i in idx[cut:]]
train_words = [word_sents[i] for i in idx[:cut]]

print(f"语料: Alice in Wonderland ONLY")
print(f"句子数: train={len(train_tagged)}, test={len(test_tagged)}")
print("样例:", train_tagged[0][:12])

语料: Alice in Wonderland ONLY
句子数: train=1461, test=163
样例: [('“', 'VB'), ('the', 'DT'), ('first', 'JJ'), ('thing', 'NN'), ('i', 'PRP'), ('’', 'VBP'), ('ve', 'JJ'), ('got', 'VBD'), ('to', 'TO'), ('do', 'VB'), (',', ','), ('”', 'VB')]


## 2. 训练 HMM 与 N-gram（同一批 Alice 训练句）

In [3]:
hmm = HiddenMarkovModel.train(train_tagged, alpha=0.1, min_word_count=2)
ngram = NgramLM.train(train_words, order=3, alpha=0.1, min_word_count=2)  # trigram

print(f"HMM:  K={len(hmm.tags)} tags, V={len(hmm.vocab)} words")
print(f"N-gram: order={ngram.order}, V={len(ngram.vocab)} words")
print(f"HMM 测试词级准确率: {hmm.accuracy(test_tagged):.3f}")

HMM:  K=41 tags, V=1425 words
N-gram: order=3, V=1427 words


HMM 测试词级准确率: 0.872


## 3. HMM 词性预测（对照用）

In [4]:
demo = test_tagged[0]
words = [w for w, _ in demo]
gold = [t for _, t in demo]
pred = hmm.tag(words)
print("词 / 金标(自动标) / HMM预测")
for w, g, p in zip(words[:20], gold[:20], pred[:20]):
    print(f"  {w:14s}  {g:5s}  {p:5s}  {'✓' if g == p else '✗'}")

词 / 金标(自动标) / HMM预测
  “               NN     NN     ✓
  as              NNP    IN     ✗
  wet             NN     VBG    ✗
  as              IN     RB     ✗
  ever            RB     RB     ✓
  ,               ,      ,      ✓
  ”               NNP    NNP    ✓
  said            VBD    VBD    ✓
  alice           NNP    NNP    ✓
  in              IN     IN     ✓
  a               DT     DT     ✓
  melancholy      JJ     JJ     ✓
  tone            NN     NN     ✓
  :               :      :      ✓
  “               NN     NN     ✓
  it              PRP    PRP    ✓
  doesn           VBZ    VBZ    ✓
  ’               JJ     JJ     ✓
  t               NN     NN     ✓
  seem            VBP    VBP    ✓


## 4. 可控生成对照（可调句子长度与 N-gram 的 N）

| 控件 | 作用 |
|---|---|
| 前缀词 | 固定开头 |
| 句子长度 | 整句词数（含前缀），HMM 与 N-gram **共用** |
| N-gram 的 N | 只用在 N-gram：用最近 $N-1$ 个词预测下一词；改 N 会按需重训 |
| 条数 / 种子 | 一次出几条、是否可复现 |
| 多阶对照 | 勾选后，同一前缀与长度下并排生成 N=2…当前N（便于看阶数影响） |


In [5]:
prefix_box = widgets.Text(
    value="alice was",
    description="前缀词",
    layout=widgets.Layout(width="min(100%, 560px)"),
    style={"description_width": "7em"},
)
length_box = widgets.IntSlider(
    value=14, min=4, max=40, step=1,
    description="句子长度",
    continuous_update=False,
    style={"description_width": "7em"},
    layout=widgets.Layout(width="min(100%, 560px)"),
)
n_order_box = widgets.IntSlider(
    value=3, min=1, max=5, step=1,
    description="N-gram 的 N",
    continuous_update=False,
    style={"description_width": "7em"},
    layout=widgets.Layout(width="min(100%, 560px)"),
)
n_box = widgets.BoundedIntText(
    value=3, min=1, max=15, description="生成条数",
    style={"description_width": "7em"},
)
seed_box = widgets.IntText(
    value=0, description="随机种子",
    style={"description_width": "7em"},
)
multi_n = widgets.Checkbox(
    value=True,
    description="多阶对照（同时生成 N=2…当前N，共用长度/前缀/种子）",
    indent=False,
)
btn = widgets.Button(description="生成对照", button_style="primary")
out = widgets.Output()

_ngram_cache = {3: ngram}

def _get_ngram(order: int) -> NgramLM:
    if order not in _ngram_cache:
        _ngram_cache[order] = NgramLM.train(
            train_words, order=order, alpha=0.1, min_word_count=2
        )
    return _ngram_cache[order]

def _fmt_words(ws):
    return " ".join(ws)

def _on_generate(_=None):
    with out:
        clear_output(wait=True)
        prefix = prefix_box.value.strip()
        length = int(length_box.value)
        n_samples = int(n_box.value)
        seed = int(seed_box.value)
        n_order = int(n_order_box.value)
        try:
            hmm_batch = hmm.generate_batch(
                prefix=prefix, length=length, n=n_samples, seed=seed
            )
            main_ng = _get_ngram(n_order)
            main_batch = main_ng.generate_batch(
                prefix=prefix, length=length, n=n_samples, seed=seed,
                stop_at_eos=False,
            )
        except ValueError as e:
            print("参数错误:", e)
            return

        print(
            f"语料=Alice ONLY | 前缀={prefix!r} | 句子长度={length} | "
            f"条数={n_samples} | 主对照 N={n_order}\n"
        )
        for i, (hp, nw) in enumerate(zip(hmm_batch, main_batch), 1):
            print(f"── 样本 [{i}] ──")
            print(f"HMM          : {_fmt_words(w for w, _ in hp)}")
            print(f"             : {' '.join(t for _, t in hp)}")
            print(f"N-gram (N={n_order}): {_fmt_words(nw)}")
            print()

        if multi_n.value:
            orders = list(range(1, n_order + 1))
            print("======== 多阶 N-gram 对照（第 1 条样本，同种子）========")
            print(f"前缀={prefix!r} | 句子长度={length}\n")
            for order in orders:
                ng = _get_ngram(order)
                one = ng.generate_batch(
                    prefix=prefix, length=length, n=1, seed=seed, stop_at_eos=False
                )[0]
                print(f"N={order}: {_fmt_words(one)}")
            print()

btn.on_click(_on_generate)
display(widgets.VBox([
    prefix_box,
    length_box,
    n_order_box,
    n_box,
    seed_box,
    multi_n,
    btn,
    out,
]))
_on_generate()


## 5. 读结果时看什么

- 先固定**句子长度**与前缀，只拧 **N**：N 太小（如 2）往往碎、跳；N 变大更贴原文搭配，也更容易在稀有前缀上变怪或重复。
- 再固定 N，只拧**句子长度**：越长越容易露出模型只会局部接词、接不住情节的问题。
- **HMM** 行看词性骨架是否顺；**N-gram** 行看用词是否像 Alice。两者都不是在写完整故事。
